# CONTEXTUAL EMBEDDING = TRUE
# NGRAM = TRUE
# MIN_COUNT = 10
# EMBEDDER = all-MiniLM-L6-v2

In [1]:
!pip install gensim top2vec


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from top2vec import Top2Vec
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

c:\Users\Allen\Documents\Python Env\environments\derp_learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(r"C:\Users\Allen\Documents\GitHub\thesis-research\combined_tweets_dataset\sample_v2\vibe_coding_combined_translated.csv")

In [4]:
df = df[["full_text_translated", "image_url"]]

In [5]:
df.head()

,full_text_translated,image_url
0,@karpathy me vibe coding and hope it can just ...,https://pbs.twimg.com/media/Gi0a82HaQAA97Q-.jpg
1,vibes capital meets vibe coding = greatest eco...,NaN
2,vibe coding https://t.co/1hHVMmunrw,https://pbs.twimg.com/media/Gi0f9fAWgAAgox6.jpg
3,can attest. vibe coding is hella fun and borde...,NaN
4,I have built a few app ideas in my spare time ...,NaN


In [6]:
def preprocess_tweet(text):
    #lower case text
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    #remove symbols
    text = re.sub(r'[^\w\s]', '', text)
    #remove numbers
    text = re.sub(r'\d+', '', text)
    #Replace all "vibe code" into vibecode
    text = re.sub(r'vibe code', 'vibecode', text)
    #Replace all "vibe coding" into vibecoding
    text = re.sub(r'vibe coding', 'vibecoding', text)
    #Relace all "vibe coded" into vibecoded
    text = re.sub(r'vibe coded', 'vibecoded', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [7]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [8]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

full_text_translated    20511
image_url                6495
dtype: int64

In [9]:
docs = df_preprocessed['full_text_translated'].tolist()
images  = df_preprocessed['image_url'].tolist()
for i in range(len(images)):
    if pd.isna(images[i]):
        images[i] = None
random.shuffle(docs)
docs[:10]

['just built the goal aligner a complete aipowered productivity tool using only vibecoding gemini api integration pdf generation smart goal alignment educational frameworks try it free',
 'may i intorduce ctclicker i created a small crypto twitter based clicker game its not my usual utility tool it was a vibecode practice run for me but maybe some of you enjoy to play a bit',
 'vibecoding with the squad',
 'vibecoding today so i can collect residual income tomorrow while im relaxing on the beach i cant share the full idea yet but i can say this im building a fully autonomous business that runs itself the future looks exciting and im all in',
 'a big reason i want to do a vibecoding app challenge ive been in marketing roles my whole career but ive never shipped and marketed a product of my own ive generated millions of views thousands of users and millions of dollars for businesses kinda wanna see if i can',
 'from a private exchange in re the article about the vibecoding boombust',
 'p

In [10]:
# embedding_model = SentenceTransformer("all-mpnet-base-v2")

umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN(min_cluster_size=15, min_samples=1)
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

representation_model = KeyBERTInspired()

In [11]:
# Top2Vec topic modeling
documents = [doc for doc in docs if isinstance(doc, str) and doc.strip()]
tokenized_documents = [doc.split() for doc in documents]

dictionary = Dictionary(tokenized_documents)
dictionary.filter_extremes(no_below=2, no_above=0.5)

print(f"tokenized documents: {tokenized_documents[:5]}")
print(f"Number of documents: {len(documents)}")
print(f"Number of unique tokens: {len(dictionary)}")



tokenized documents: [['just', 'built', 'the', 'goal', 'aligner', 'a', 'complete', 'aipowered', 'productivity', 'tool', 'using', 'only', 'vibecoding', 'gemini', 'api', 'integration', 'pdf', 'generation', 'smart', 'goal', 'alignment', 'educational', 'frameworks', 'try', 'it', 'free'], ['may', 'i', 'intorduce', 'ctclicker', 'i', 'created', 'a', 'small', 'crypto', 'twitter', 'based', 'clicker', 'game', 'its', 'not', 'my', 'usual', 'utility', 'tool', 'it', 'was', 'a', 'vibecode', 'practice', 'run', 'for', 'me', 'but', 'maybe', 'some', 'of', 'you', 'enjoy', 'to', 'play', 'a', 'bit'], ['vibecoding', 'with', 'the', 'squad'], ['vibecoding', 'today', 'so', 'i', 'can', 'collect', 'residual', 'income', 'tomorrow', 'while', 'im', 'relaxing', 'on', 'the', 'beach', 'i', 'cant', 'share', 'the', 'full', 'idea', 'yet', 'but', 'i', 'can', 'say', 'this', 'im', 'building', 'a', 'fully', 'autonomous', 'business', 'that', 'runs', 'itself', 'the', 'future', 'looks', 'exciting', 'and', 'im', 'all', 'in'], ['a

In [12]:
print(dictionary.token2id)

{'a': 0, 'aipowered': 1, 'alignment': 2, 'api': 3, 'built': 4, 'complete': 5, 'educational': 6, 'frameworks': 7, 'free': 8, 'gemini': 9, 'generation': 10, 'goal': 11, 'integration': 12, 'it': 13, 'just': 14, 'only': 15, 'pdf': 16, 'productivity': 17, 'smart': 18, 'tool': 19, 'try': 20, 'using': 21, 'based': 22, 'bit': 23, 'but': 24, 'clicker': 25, 'created': 26, 'crypto': 27, 'enjoy': 28, 'for': 29, 'game': 30, 'i': 31, 'its': 32, 'may': 33, 'maybe': 34, 'me': 35, 'my': 36, 'not': 37, 'of': 38, 'play': 39, 'practice': 40, 'run': 41, 'small': 42, 'some': 43, 'to': 44, 'twitter': 45, 'usual': 46, 'utility': 47, 'vibecode': 48, 'was': 49, 'you': 50, 'squad': 51, 'with': 52, 'all': 53, 'and': 54, 'autonomous': 55, 'beach': 56, 'building': 57, 'business': 58, 'can': 59, 'cant': 60, 'collect': 61, 'exciting': 62, 'full': 63, 'fully': 64, 'future': 65, 'idea': 66, 'im': 67, 'in': 68, 'income': 69, 'itself': 70, 'looks': 71, 'on': 72, 'relaxing': 73, 'runs': 74, 'say': 75, 'share': 76, 'so': 7

In [13]:
import top2vec
print(top2vec.__version__)

1.0.36


In [14]:
top2vec_model = Top2Vec(
    documents=documents,
    contextual_top2vec=True,
    ngram_vocab=True,
    min_count=10,
    embedding_model='all-MiniLM-L6-v2',    
    speed='learn',
    workers=2,
    keep_documents=True,
    verbose=True,
 )

topic_words, topic_word_scores, topic_nums = top2vec_model.get_topics()



2026-04-12 15:14:31,480 - top2vec - INFO - Pre-processing documents for training
2026-04-12 15:14:33,069 - top2vec - INFO - Creating vocabulary embedding
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6370.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding vocabulary: 100%|██████████| 41/41 [00:04<00:00,  8.66it/s]
2026-04-12 15:14:41,139 - top2vec - INFO - Create contextualized document embeddings
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6355.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored whe

In [15]:
topic_sizes, topic_ids = top2vec_model.get_topic_sizes()
pd.DataFrame({
    "topic_id": topic_ids,
    "size": topic_sizes
}).sort_values("size", ascending=False).head(10)

,topic_id,size
0,1,671700
1,0,8093


In [16]:
print(len(topic_sizes))

2


In [17]:
# Coherence metrics for Top2Vec topics (C_V and NPMI)
topic_pairs = []
for topic_num, topic in zip(topic_nums, topic_words):
    cleaned_topic_terms = []
    for term in topic[:10]:
        # ngram terms like "vibe coding" are split so they can be matched to dictionary tokens
        for tok in str(term).split():
            if tok in dictionary.token2id:
                cleaned_topic_terms.append(tok)
    # keep token order, drop duplicates
    cleaned_topic_terms = list(dict.fromkeys(cleaned_topic_terms))
    if len(cleaned_topic_terms) >= 2:
        topic_pairs.append((topic_num, cleaned_topic_terms))

topics_for_coherence = [words for _, words in topic_pairs]
topic_nums_for_coherence = [topic_num for topic_num, _ in topic_pairs]

if not topics_for_coherence:
    raise ValueError("No valid topics left for coherence calculation. Try lowering dictionary filtering or increasing topic words.")

coherence_cv_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_v',
)

coherence_npmi_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_npmi',
)

coherence_cv = coherence_cv_model.get_coherence()
coherence_npmi = coherence_npmi_model.get_coherence()

print(f"Total topics used for coherence: {len(topics_for_coherence)}")
print(f"C_V coherence: {coherence_cv:.4f}")
print(f"NPMI coherence: {coherence_npmi:.4f}")

topic_coherence = pd.DataFrame({
    'topic_num': topic_nums_for_coherence,
    'topic_words': [' '.join(words) for words in topics_for_coherence],
    'c_v': coherence_cv_model.get_coherence_per_topic(),
    'npmi': coherence_npmi_model.get_coherence_per_topic(),
})

topic_coherence.sort_values('c_v', ascending=False).head(10)

Total topics used for coherence: 2
C_V coherence: 0.4163
NPMI coherence: -0.2312


,topic_num,topic_words,c_v,npmi
0,0,dance clubfor vibe designing marketing vibecod...,0.460333,-0.267261
1,1,vibe debugging marketing engineering designing...,0.372224,-0.195152


In [26]:
# get all topic words values
topic_words_values = topic_coherence.sort_values('c_v', ascending=False).head(10)['topic_words'].values
print(topic_words_values)


['dance clubfor vibe designing marketing vibecoded projects investing vibecode camp punk rock engineering living debugging'
 'vibe debugging marketing engineering designing vibecoded projects investing vibecode cleanup ltcodinggt camp living']


In [18]:
# Save model
top2vec_model.save("top2vec_contextual_ngram_c10.model")

# Load model (di masa depan)
# from top2vec import Top2Vec
top2vec_model = Top2Vec.load("top2vec_contextual_ngram_c10.model")



# Load di kemudian hari
# embeddings = np.load("top2vec_contextual_ngram_c10_embeddings.npy")


# Simpan Dictionary
dictionary.save("top2vec_contextual_ngram_c10.dict")

# Simpan tokenized_documents (menggunakan pickle karena list of lists)
import pickle
with open("top2vec_contextual_ngram_c10_tokenized_docs.pkl", "wb") as f:
    pickle.dump(tokenized_documents, f)
    
    
    
# Simpan ringkasan topik dan metriknya
topic_coherence.to_csv("top2vec_contextual_ngram_c10_metrics_summary.csv", index=False)

# Simpan representasi topik (10 kata teratas)
top_words_df = pd.DataFrame(topic_words)
top_words_df.to_csv("top2vec_contextual_ngram_c10_topic_words.csv", index=False)